step 1 - create a folder structure

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dbwork1.project")
# Tables will be created as:
# dbwork1.project.source
# dbwork1.project.destination

DataFrame[]

In [0]:
%sql SHOW CATALOGS

catalog
dbwork1
samples
system


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbwork1.default.dbwork1_volume;

In [0]:

%sql SHOW SCHEMAS IN dbwork1


databaseName
default
information_schema
marketing
project


In [0]:
 %sql SHOW VOLUMES IN dbwork1.default


database,volume_name
default,dbwork1_volume
default,project


In [0]:
dbutils.fs.mkdirs("/Volumes/dbwork1/default/project/Source")
dbutils.fs.mkdirs("/Volumes/dbwork1/default/project/Destination")

True

In [0]:
dbutils.fs.ls("/Volumes/dbwork1/default/project")

[FileInfo(path='dbfs:/Volumes/dbwork1/default/project/Destination/', name='Destination/', size=0, modificationTime=1775196763000),
 FileInfo(path='dbfs:/Volumes/dbwork1/default/project/Source/', name='Source/', size=0, modificationTime=1775196763000)]

uploaded the files into source folder

step2 - load csv files

In [0]:
dbutils.fs.ls("/Volumes/dbwork1/default/project/Source")

[FileInfo(path='dbfs:/Volumes/dbwork1/default/project/Source/Categories.csv', name='Categories.csv', size=65, modificationTime=1775198402000),
 FileInfo(path='dbfs:/Volumes/dbwork1/default/project/Source/Order_Item.csv', name='Order_Item.csv', size=124, modificationTime=1775198401000),
 FileInfo(path='dbfs:/Volumes/dbwork1/default/project/Source/Orders.csv', name='Orders.csv', size=123, modificationTime=1775198401000),
 FileInfo(path='dbfs:/Volumes/dbwork1/default/project/Source/Products.csv', name='Products.csv', size=109, modificationTime=1775198401000)]

In [0]:
orders_data=spark.read.csv("/Volumes/dbwork1/default/project/Source/Orders.csv",header=True,inferSchema=True)
order_item_data=spark.read.csv("/Volumes/dbwork1/default/project/Source/Order_Item.csv",header=True,inferSchema=True)
product_data=spark.read.csv("/Volumes/dbwork1/default/project/Source/Products.csv",header=True,inferSchema=True)
categories_data=spark.read.csv("/Volumes/dbwork1/default/project/Source/Categories.csv",header=True,inferSchema=True)


step 3 - show/verfy data

In [0]:
orders_data.show()

+--------+-----------+----------+
|order_id|customer_id|order_date|
+--------+-----------+----------+
|       1|        101|2022-01-05|
|       2|        102|2022-02-10|
|       3|        103|2022-03-15|
|       4|        101|2022-04-20|
|       5|        104|2022-05-25|
+--------+-----------+----------+



In [0]:
order_item_data.show()

+--------+----------+--------+------------+
|order_id|product_id|quantity|total_amount|
+--------+----------+--------+------------+
|       1|         1|       5|       100.0|
|       1|         2|       3|        75.0|
|       2|         3|       2|        50.0|
|       3|         2|       4|       100.0|
|       4|         1|       7|       140.0|
|       5|         4|       2|        60.0|
+--------+----------+--------+------------+



In [0]:
product_data.show()

+----------+------------+-----------+
|product_id|product_name|category_id|
+----------+------------+-----------+
|         1|      Laptop|          1|
|         2|  Smartphone|          1|
|         3|      Tablet|          1|
|         4|  Headphones|          2|
|         5|  Smartwatch|          2|
+----------+------------+-----------+



In [0]:
categories_data.show()

+-----------+-------------+
|category_id|category_name|
+-----------+-------------+
|          1|  Electronics|
|          2|  Accessories|
|          3|         Home|
+-----------+-------------+



step 4 - data transformation

#Perform inner joins to consolidate sales data with product and category details

In [0]:

#1. What is the total revenue for each product category?
#2. What is the total revenue for each product?
#3. What is the total revenue for each customer?
#4. What is the total revenue for each day?
#5. What is the total revenue for each month?
#6. What is the total revenue for each quarter?
#7. What is the total revenue for each year?
#8. What is the total revenue for each product category for each day?
#9. What is the total revenue for each product category for each month?
#10. What is the total revenue for each product category for each quarter?
#11. What is the total revenue for each product category for each year?
#12. What is the total revenue for each product for each day?
#13. What is the total revenue for each product for each month?
#14. What is the total revenue for each product for each quarter?
#15. What is the total revenue for each product for each year?
#16. What is the total revenue for each customer for each day?
#17. What is the total revenue for each customer for each month?
#18. What is the total revenue for each customer for each quarter?
#19. What is the total revenue for each customer for each year?
#20. What is the total revenue for each product category for each customer?
#21. What is the total revenue for each product for each customer?
#22. What is the total revenue for each day for each customer?
#23. What is the total revenue for each month for each customer?
#24. What is the total revenue for each quarter for each customer?
#25. What is the total revenue for each year for each customer?
#26. What is the total revenue for each product category for each day for each customer?
#27. What is the total revenue for each product for each day for each customer?
#28. What is the total revenue for each product category for each month for each customer?
#29. What is the total revenue for each product for each month for each customer

In [0]:
from pyspark.sql.functions import *

joined_data=(
    orders_data.join(order_item_data,"order_id","inner")
    .join(product_data,"product_id","inner")
    .join(categories_data,"category_id","inner")
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_date"),
        col("product_name"),
        col("category_name"),
        col("quantity"),
        col("total_amount")
    )
)

In [0]:
joined_data.show()

+--------+-----------+----------+------------+-------------+--------+------------+
|order_id|customer_id|order_date|product_name|category_name|quantity|total_amount|
+--------+-----------+----------+------------+-------------+--------+------------+
|       1|        101|2022-01-05|      Laptop|  Electronics|       5|       100.0|
|       1|        101|2022-01-05|  Smartphone|  Electronics|       3|        75.0|
|       2|        102|2022-02-10|      Tablet|  Electronics|       2|        50.0|
|       3|        103|2022-03-15|  Smartphone|  Electronics|       4|       100.0|
|       4|        101|2022-04-20|      Laptop|  Electronics|       7|       140.0|
|       5|        104|2022-05-25|  Headphones|  Accessories|       2|        60.0|
+--------+-----------+----------+------------+-------------+--------+------------+



In [0]:
cust_df=joined_data.groupby('customer_id').agg(sum('quantity').alias('total_quantity'),sum('total_amount').alias('total_amount'))




In [0]:
cust_df.show()


+-----------+--------------+------------+
|customer_id|total_quantity|total_amount|
+-----------+--------------+------------+
|        101|            15|       315.0|
|        103|             4|       100.0|
|        102|             2|        50.0|
|        104|             2|        60.0|
+-----------+--------------+------------+



In [0]:
joined_data.write.csv("/Volumes/dbwork1/default/project/Destination/sales_report.csv")
cust_df.write.csv("/Volumes/dbwork1/default/project/Destination/customer.csv")
display(joined_data)
display(cust_df)

order_id,customer_id,order_date,product_name,category_name,quantity,total_amount
1,101,2022-01-05,Laptop,Electronics,5,100.0
1,101,2022-01-05,Smartphone,Electronics,3,75.0
2,102,2022-02-10,Tablet,Electronics,2,50.0
3,103,2022-03-15,Smartphone,Electronics,4,100.0
4,101,2022-04-20,Laptop,Electronics,7,140.0
5,104,2022-05-25,Headphones,Accessories,2,60.0


customer_id,total_quantity,total_amount
101,15,315.0
103,4,100.0
102,2,50.0
104,2,60.0
